# 拒绝抽样法生成 Beta 分布随机数

> 对应代码：`MathStatsCode/code_in_notes/Chap.5/rejection_beta.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：以均匀分布为工具分布，用拒绝抽样法从 Beta(4,2) 分布中生成随机数。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.5`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.5")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


准备10000个候选观测（`set obs 10000`），设定种子保证可复现；用 `local` 定义两个局部宏 `alpha=4`、`beta=2` 存放 Beta 分布的参数。注意**候选样本量要远大于目标样本量**：拒绝抽样会产生大量被拒绝的候选点，最终接受的样本数取决于接受率。


In [2]:
%%stata
clear
set obs 10000
set seed 505
local alpha=4
local beta=2



. clear

. set obs 10000
Number of observations (_N) was 0, now 10,000.

. set seed 505

. local alpha=4

. local beta=2

. 


**求密度函数极值**

计算包络常数 M：Beta(4,2) 的密度函数在众数 x*=(α-1)/(α+β-2)=3/4 处取得最大值 `betaden(α,β,x*)`，即 M。随后 `gen u=runiform()*M` 在 [0,M] 上均匀地产生“纵坐标”，`gen x=runiform()` 在 [0,1] 上均匀地产生候选抽样点——二者的联合分布是密度曲线外的矩形区域上的均匀分布。


In [3]:
%%stata
local M=betaden(`alpha',`beta',(`alpha'-1)/(`alpha'+`beta'-2))
gen u=runiform()*`M'
gen x=runiform()



. local M=betaden(`alpha',`beta',(`alpha'-1)/(`alpha'+`beta'-2))

. gen u=runiform()*`M'

. gen x=runiform()

. 


**计算密度函数**

`gen density=betaden(alpha,beta,x)`：用 Stata 内置的 Beta 密度函数 `betaden(α,β,x)` 计算每个候选点处的目标密度值，作为接受判据。


In [4]:
%%stata
gen density=betaden(`alpha',`beta',x)


**抽样**

`keep if u<density`：**接受-拒绝判别**。候选点 (x,u) 落在密度曲线下方就保留、否则删除。几何上，矩形区域内的均匀点落在曲线下的比例等于密度曲线下的面积（=1），而保留点的 x 边际分布正是密度函数本身——这就是拒绝抽样成立的直观理由。


In [5]:
%%stata
keep if u<density


(5,236 observations deleted)


**查看抽样的描述性统计**

`su x`：对保留下来的样本做描述性统计。由 Beta(4,2) 的理论均值 α/(α+β)=2/3≈0.667，应能观察到样本均值非常接近这一数值。


In [6]:
%%stata
su x



    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
           x |      4,764    .6704633    .1772083   .0142275   .9924354
